# Top-N and Ranking Patterns

## Quick notes
### What this covers
- Top-N overall and top-N per group with `ROW_NUMBER`, `RANK`, `DENSE_RANK`.
- Ties and tie-breakers, the "nth highest salary" question solved three ways.
- Latest row per group, first/last per group, de-duplicating and finding duplicates.
- Bottom-N, "top N plus Other", percentile buckets, leaderboards.
- Greatest-n-per-group without window functions, and keyset pagination.
### Key points
- "Top N" is really two questions: how do you rank, and what happens on a tie?
- `ROW_NUMBER` gives exactly N rows (when there are at least N). `RANK` and `DENSE_RANK` may give more than N because tied rows share a rank.
- Rank inside a CTE or subquery, then filter on the rank in the outer query. You cannot filter on a window function in `WHERE`.
- Always add a unique column (like `id`) as the last `ORDER BY` key so the answer is repeatable.
- Without window functions you can still do it: correlated subquery, join to `MAX()`, or `NOT EXISTS`.
### Gotchas (the part worth revising twice)
- `LIMIT` without `ORDER BY` returns "some" rows. `LIMIT` with a tied `ORDER BY` returns an arbitrary choice among the ties.
- `LIMIT 1 OFFSET n-1` for the nth highest salary breaks with duplicates (it returns the nth row, not the nth distinct value).
- "Latest per group" with `WHERE date = (SELECT MAX(date) ...)` returns several rows if two rows share the latest date.
- `NOT IN (subquery)` returns nothing if the subquery has a `NULL`. Prefer `NOT EXISTS`.
- `OFFSET` pagination gets slower the deeper you go and skips or repeats rows when data changes. Keyset pagination fixes both.
### When it matters
- Leaderboards, "best-selling product per category", "latest status per ticket", "second highest salary" interview questions, clean-up of duplicate imports.
### One-line recap example
```sql
WITH r AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC, id) AS rn FROM emp)
SELECT * FROM r WHERE rn <= 2;   -- top 2 per department
```

## 01. Seed data
- `emp`: employees with deliberate salary ties (Bob and Gus; Carla and Kim; Ivy and Jo).
- `orders`: customer orders, with two orders by the same customer on the same day.
- `contact`: a messy import with duplicate emails.
- Window function basics are in `03-Window-Functions/01-Window-Functions`. This notebook is about the patterns built on top.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE emp (id INTEGER PRIMARY KEY, name TEXT, dept TEXT, salary INTEGER);
INSERT INTO emp VALUES
 (1,'Alice','Eng',9500),(2,'Bob','Eng',7200),(3,'Gus','Eng',7200),(4,'Hana','Eng',6000),
 (5,'Carla','Sales',6800),(6,'Dev','Sales',6100),(7,'Eve','Sales',5800),(11,'Kim','Sales',6800),
 (8,'Finn','Mkt',7500),(9,'Ivy','Mkt',5000),(10,'Jo','Mkt',5000);

CREATE TABLE orders (id INTEGER PRIMARY KEY, customer TEXT, order_date TEXT, amount INTEGER);
INSERT INTO orders VALUES
 (1,'Asha','2024-01-05',120),(2,'Asha','2024-02-10',80),(3,'Asha','2024-02-10',200),
 (4,'Ben','2024-01-20',50),(5,'Ben','2024-03-01',300),
 (6,'Cy','2024-01-02',75),(7,'Cy','2024-01-30',75),(8,'Cy','2024-02-14',60),(9,'Dee','2024-03-09',500);

CREATE TABLE contact (id INTEGER PRIMARY KEY, email TEXT, name TEXT, created_at TEXT);
INSERT INTO contact VALUES
 (1,'a@x.com','Asha','2024-01-01'),(2,'b@x.com','Ben','2024-01-02'),
 (3,'a@x.com','Asha K','2024-01-09'),(4,'c@x.com','Cy','2024-01-10'),
 (5,'b@x.com','Ben R','2024-01-15'),(6,'a@x.com','A. Kumar','2024-01-20');
SELECT id, name, dept, salary FROM emp ORDER BY id;

id | name  | dept  | salary
---+-------+-------+-------
1  | Alice | Eng   | 9500  
2  | Bob   | Eng   | 7200  
3  | Gus   | Eng   | 7200  
4  | Hana  | Eng   | 6000  
5  | Carla | Sales | 6800  
6  | Dev   | Sales | 6100  
7  | Eve   | Sales | 5800  
8  | Finn  | Mkt   | 7500  
9  | Ivy   | Mkt   | 5000  
10 | Jo    | Mkt   | 5000  
11 | Kim   | Sales | 6800  
(11 rows)



## 02. Top-N overall: ORDER BY + LIMIT
- The simplest top-N: sort, then cut. Fine when exact N rows is what you want.
- Problem: Bob and Gus both earn 7200. "Top 3" has two valid answers, and `LIMIT` picks one without telling you.
- Adding `id` as a last sort key does not remove the unfairness, but it makes the result repeatable.

In [3]:
%%sql
-- ties at 7200: which of Bob / Gus is third? Alice, Finn, then one of them
SELECT name, salary FROM emp ORDER BY salary DESC, id LIMIT 3;
-- the tie-break flipped, a different but equally "valid" top 3
SELECT name, salary FROM emp ORDER BY salary DESC, id DESC LIMIT 3;

name  | salary
------+-------
Alice | 9500  
Finn  | 7500  
Bob   | 7200  
(3 rows)

name  | salary
------+-------
Alice | 9500  
Finn  | 7500  
Gus   | 7200  
(3 rows)



## 03. ROW_NUMBER vs RANK vs DENSE_RANK
- `ROW_NUMBER`: 1,2,3,4... never ties, ties are split arbitrarily (or by your tie-breaker).
- `RANK`: tied rows share a rank, the next rank skips (1,2,2,4).
- `DENSE_RANK`: tied rows share a rank, no skipping (1,2,2,3).
- Pick by the business question: "exactly 3 rows" = `ROW_NUMBER`, "everybody in the top 3 places" = `RANK`, "top 3 distinct values" = `DENSE_RANK`.

In [4]:
%%sql
SELECT name, salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC, id) AS row_num,
       RANK()       OVER (ORDER BY salary DESC)     AS rnk,
       DENSE_RANK() OVER (ORDER BY salary DESC)     AS dense
FROM emp ORDER BY salary DESC, id;

name  | salary | row_num | rnk | dense
------+--------+---------+-----+------
Alice | 9500   | 1       | 1   | 1    
Finn  | 7500   | 2       | 2   | 2    
Bob   | 7200   | 3       | 3   | 3    
Gus   | 7200   | 4       | 3   | 3    
Carla | 6800   | 5       | 5   | 4    
Kim   | 6800   | 6       | 5   | 4    
Dev   | 6100   | 7       | 7   | 5    
Hana  | 6000   | 8       | 8   | 6    
Eve   | 5800   | 9       | 9   | 7    
Ivy   | 5000   | 10      | 10  | 8    
Jo    | 5000   | 11      | 10  | 8    
(11 rows)



## 04. Top-N filter: which function gives how many rows?
- Same question ("top 3"), three answers. Count the rows.
- Remember to filter in an outer query: the rank does not exist yet in the inner `WHERE`.

In [5]:
%%sql
WITH r AS (
  SELECT name, salary,
         ROW_NUMBER() OVER (ORDER BY salary DESC, id) AS rn,
         RANK()       OVER (ORDER BY salary DESC)     AS rk,
         DENSE_RANK() OVER (ORDER BY salary DESC)     AS dr
  FROM emp)
SELECT 'row_number' AS method, COUNT(*) AS rows_returned FROM r WHERE rn <= 3
UNION ALL SELECT 'rank',       COUNT(*) FROM r WHERE rk <= 3
UNION ALL SELECT 'dense_rank', COUNT(*) FROM r WHERE dr <= 3
ORDER BY method;

method     | rows_returned
-----------+--------------
dense_rank | 4            
rank       | 4            
row_number | 3            
(3 rows)



In [6]:
%%sql
-- the actual people for RANK <= 3: ties at 7200 are both kept (4 rows)
WITH r AS (SELECT name, salary, RANK() OVER (ORDER BY salary DESC) AS rk FROM emp)
SELECT name, salary, rk FROM r WHERE rk <= 3 ORDER BY rk, name;

name  | salary | rk
------+--------+---
Alice | 9500   | 1 
Finn  | 7500   | 2 
Bob   | 7200   | 3 
Gus   | 7200   | 3 
(4 rows)



## 05. Tie-breakers: make the result deterministic
- If two rows tie on the sort key, the database may return either first, and the answer can change after an index is added.
- Add a business tie-breaker (earliest hire, alphabetical, lowest id) as the next `ORDER BY` key.
- The last key should be unique (usually the primary key), so no ties remain.

In [7]:
%%sql
SELECT name, salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC, name) AS by_name,
       ROW_NUMBER() OVER (ORDER BY salary DESC, id DESC) AS by_id_desc
FROM emp WHERE dept = 'Eng' ORDER BY by_name;

name  | salary | by_name | by_id_desc
------+--------+---------+-----------
Alice | 9500   | 1       | 1         
Bob   | 7200   | 2       | 3         
Gus   | 7200   | 3       | 2         
Hana  | 6000   | 4       | 4         
(4 rows)



## 06. Top-N per group
- `PARTITION BY dept` restarts the numbering for every department.
- `ROW_NUMBER` gives exactly 2 per department (if they have 2). `RANK` keeps ties, so Eng and Mkt show 3 rows for "top 2" (two people tie for 2nd), while Sales shows 2 (Carla and Kim tie for 1st, and Dev is rank 3).

In [8]:
%%sql
-- exactly 2 per department
WITH r AS (
  SELECT dept, name, salary,
         ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC, id) AS rn
  FROM emp)
SELECT dept, name, salary, rn FROM r WHERE rn <= 2 ORDER BY dept, rn;

dept  | name  | salary | rn
------+-------+--------+---
Eng   | Alice | 9500   | 1 
Eng   | Bob   | 7200   | 2 
Mkt   | Finn  | 7500   | 1 
Mkt   | Ivy   | 5000   | 2 
Sales | Carla | 6800   | 1 
Sales | Kim   | 6800   | 2 
(6 rows)



In [9]:
%%sql
-- 2 places per department, ties kept
WITH r AS (
  SELECT dept, name, salary,
         RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rk
  FROM emp)
SELECT dept, name, salary, rk FROM r WHERE rk <= 2 ORDER BY dept, rk, name;

dept  | name  | salary | rk
------+-------+--------+---
Eng   | Alice | 9500   | 1 
Eng   | Bob   | 7200   | 2 
Eng   | Gus   | 7200   | 2 
Mkt   | Finn  | 7500   | 1 
Mkt   | Ivy   | 5000   | 2 
Mkt   | Jo    | 5000   | 2 
Sales | Carla | 6800   | 1 
Sales | Kim   | 6800   | 1 
(8 rows)



## 07. Nth highest salary, way 1: LIMIT / OFFSET
- Skip N-1 rows, take 1. Short, but it counts rows, not distinct values.
- Salaries sorted: 9500, 7500, 7200, 7200, 6800, 6800 ... The "3rd highest salary" by this method is 7200 (fine), the "4th" is 7200 again (a duplicate, not a new value).
- Fix: `SELECT DISTINCT salary` first, then `LIMIT/OFFSET`.

In [10]:
%%sql
-- 4th row, not the 4th distinct value
SELECT salary FROM emp ORDER BY salary DESC LIMIT 1 OFFSET 3;
-- 4th distinct value
SELECT salary FROM (SELECT DISTINCT salary FROM emp) ORDER BY salary DESC LIMIT 1 OFFSET 3;

salary
------
7200  
(1 row)

salary
------
6800  
(1 row)



## 08. Nth highest salary, way 2: subquery (no window)
- "The Nth highest value has exactly N-1 distinct values above it."
- Count distinct larger salaries in a correlated subquery. Works on any database, but is O(n squared) without an index.
- Wrap in `SELECT MAX(...)` so you get one `NULL` row (not zero rows) when there is no Nth value.

In [11]:
%%sql
SELECT MAX(salary) AS third_highest
FROM emp e
WHERE (SELECT COUNT(DISTINCT salary) FROM emp WHERE salary > e.salary) = 2;

third_highest
-------------
7200         
(1 row)



In [12]:
%%sql
-- 20th highest does not exist: one row with NULL (thanks to MAX), not an empty result
SELECT MAX(salary) AS twentieth_highest
FROM emp e
WHERE (SELECT COUNT(DISTINCT salary) FROM emp WHERE salary > e.salary) = 19;

twentieth_highest
-----------------
NULL             
(1 row)



## 09. Nth highest salary, way 3: DENSE_RANK
- The cleanest and most flexible: rank distinct values, keep rank = N.
- It also does "Nth highest per department" with one extra `PARTITION BY`.
- Returns everybody who earns that salary. Use `SELECT DISTINCT salary` if you only want the value.

In [13]:
%%sql
WITH r AS (SELECT name, salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS dr FROM emp)
SELECT name, salary FROM r WHERE dr = 3 ORDER BY name;

name | salary
-----+-------
Bob  | 7200  
Gus  | 7200  
(2 rows)



In [14]:
%%sql
-- 2nd highest per department
WITH r AS (
  SELECT dept, name, salary,
         DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dr
  FROM emp)
SELECT dept, name, salary FROM r WHERE dr = 2 ORDER BY dept, name;

dept  | name | salary
------+------+-------
Eng   | Bob  | 7200  
Eng   | Gus  | 7200  
Mkt   | Ivy  | 5000  
Mkt   | Jo   | 5000  
Sales | Dev  | 6100  
(5 rows)



## 10. Latest row per group
- "Most recent order per customer." Rank by date descending inside each customer, keep rank 1.
- Asha has two orders on 2024-02-10 (ids 2 and 3). A tie-breaker (`id DESC`) decides which one is "latest".
- See also `01-Basics` for date text rules (ISO `YYYY-MM-DD` sorts correctly as text).

In [15]:
%%sql
WITH r AS (
  SELECT customer, order_date, amount, id,
         ROW_NUMBER() OVER (PARTITION BY customer ORDER BY order_date DESC, id DESC) AS rn
  FROM orders)
SELECT customer, order_date, amount, id FROM r WHERE rn = 1 ORDER BY customer;

customer | order_date | amount | id
---------+------------+--------+---
Asha     | 2024-02-10 | 200    | 3 
Ben      | 2024-03-01 | 300    | 5 
Cy       | 2024-02-14 | 60     | 8 
Dee      | 2024-03-09 | 500    | 9 
(4 rows)



## 11. First and last per group in one pass
- Want both the first and last value for each group? `FIRST_VALUE` and `LAST_VALUE` over the same window, then `DISTINCT`.
- `LAST_VALUE` needs a full frame (`ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`), otherwise it just returns the current row (see `03-Window-Functions/01-Window-Functions`).
- Alternative: aggregate with `MIN`/`MAX` on dates, then join back (section 17).

In [16]:
%%sql
SELECT DISTINCT customer,
       FIRST_VALUE(amount) OVER w AS first_amount,
       LAST_VALUE(amount)  OVER w AS last_amount,
       COUNT(*) OVER (PARTITION BY customer) AS n_orders
FROM orders
WINDOW w AS (PARTITION BY customer ORDER BY order_date, id
             ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
ORDER BY customer;

customer | first_amount | last_amount | n_orders
---------+--------------+-------------+---------
Asha     | 120          | 200         | 3       
Ben      | 50           | 300         | 2       
Cy       | 75           | 60          | 3       
Dee      | 500          | 500         | 1       
(4 rows)



## 12. Find duplicates
- Group by what makes a row "the same" (here, the email), keep groups with `HAVING COUNT(*) > 1`.
- To see the actual duplicate rows (not just the key), use a window count and filter.

In [17]:
%%sql
SELECT email, COUNT(*) AS copies FROM contact GROUP BY email HAVING COUNT(*) > 1 ORDER BY email;

email   | copies
--------+-------
a@x.com | 3     
b@x.com | 2     
(2 rows)



In [18]:
%%sql
-- every row that has a twin, with its position in the group
WITH c AS (
  SELECT *, COUNT(*) OVER (PARTITION BY email) AS copies,
            ROW_NUMBER() OVER (PARTITION BY email ORDER BY created_at, id) AS nth
  FROM contact)
SELECT id, email, name, created_at, nth FROM c WHERE copies > 1 ORDER BY email, nth;

id | email   | name     | created_at | nth
---+---------+----------+------------+----
1  | a@x.com | Asha     | 2024-01-01 | 1  
3  | a@x.com | Asha K   | 2024-01-09 | 2  
6  | a@x.com | A. Kumar | 2024-01-20 | 3  
2  | b@x.com | Ben      | 2024-01-02 | 1  
5  | b@x.com | Ben R    | 2024-01-15 | 2  
(5 rows)



## 13. De-duplicate: keep one row per key
- Rank the copies, keep `rn = 1` (the earliest here). Everything else is the duplicate.
- To actually delete, `DELETE ... WHERE id IN (SELECT id ... WHERE rn > 1)`. Do it in a transaction and look first (see `06-Data-Modification`).
- Which one to keep is a business decision: oldest, newest, most complete row. Put it in the `ORDER BY`.

In [19]:
%%sql
-- preview: the rows that would be deleted
WITH c AS (SELECT id, email, ROW_NUMBER() OVER (PARTITION BY email ORDER BY created_at, id) AS rn FROM contact)
SELECT id, email FROM c WHERE rn > 1 ORDER BY id;

id | email  
---+--------
3  | a@x.com
5  | b@x.com
6  | a@x.com
(3 rows)



In [20]:
%%sql
BEGIN;
DELETE FROM contact WHERE id IN (
  SELECT id FROM (SELECT id, ROW_NUMBER() OVER (PARTITION BY email ORDER BY created_at, id) AS rn FROM contact)
  WHERE rn > 1);
SELECT id, email, name FROM contact ORDER BY id;
ROLLBACK;
SELECT COUNT(*) AS rows_after_rollback FROM contact;

id | email   | name
---+---------+-----
1  | a@x.com | Asha
2  | b@x.com | Ben 
4  | c@x.com | Cy  
(3 rows)

rows_after_rollback
-------------------
6                  
(1 row)



## 14. Rank within percentile buckets
- `NTILE(n)` splits the rows into `n` roughly equal buckets (4 = quartiles). Rank again inside each bucket for "top earner of each quartile".
- With 11 employees and 4 buckets, the first 3 buckets get 3 rows and the last gets 2 (extra rows go to the early buckets).
- `NTILE` does not respect ties: Bob and Gus both earn 7200 but land in quartiles 1 and 2 below. Use `PERCENT_RANK` if that matters.

In [21]:
%%sql
WITH q AS (
  SELECT name, salary, NTILE(4) OVER (ORDER BY salary DESC, id) AS quartile FROM emp)
SELECT quartile, name, salary,
       RANK() OVER (PARTITION BY quartile ORDER BY salary DESC) AS rank_in_quartile
FROM q ORDER BY quartile, rank_in_quartile, name;

quartile | name  | salary | rank_in_quartile
---------+-------+--------+-----------------
1        | Alice | 9500   | 1               
1        | Finn  | 7500   | 2               
1        | Bob   | 7200   | 3               
2        | Gus   | 7200   | 1               
2        | Carla | 6800   | 2               
2        | Kim   | 6800   | 2               
3        | Dev   | 6100   | 1               
3        | Hana  | 6000   | 2               
3        | Eve   | 5800   | 3               
4        | Ivy   | 5000   | 1               
4        | Jo    | 5000   | 1               
(11 rows)



## 15. Bottom-N
- Flip the sort direction. Same patterns, same tie questions.
- Handy trick: `ORDER BY salary ASC` for the lowest, but keep your tie-breaker sensible so the result reads well.
- Watch `NULL`s: in SQLite, `NULL` sorts first in `ASC`. A "lowest salary" list may fill with unknowns. Filter `WHERE salary IS NOT NULL` if needed.

In [22]:
%%sql
-- lowest salary per department, ties kept (Mkt returns both Ivy and Jo)
WITH r AS (
  SELECT dept, name, salary, DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary ASC) AS dr
  FROM emp)
SELECT dept, name, salary FROM r WHERE dr <= 1 ORDER BY dept, name;

dept  | name | salary
------+------+-------
Eng   | Hana | 6000  
Mkt   | Ivy  | 5000  
Mkt   | Jo   | 5000  
Sales | Eve  | 5800  
(4 rows)



In [23]:
%%sql
-- NULLs sort first in ASC (SQLite)
SELECT x FROM (SELECT 5 AS x UNION ALL SELECT NULL UNION ALL SELECT 2) ORDER BY x ASC;

x   
----
NULL
2   
5   
(3 rows)



## 16. "Top N plus Other" bucket
- Reports often show the top 3 items and lump the rest into "Other" so the total still adds up.
- Rank the groups, then `CASE WHEN rank <= 3 THEN name ELSE 'Other' END`, and group again.
- Here: revenue per customer, top 2 customers named, the rest as "Other".

In [24]:
%%sql
WITH t AS (
  SELECT customer, SUM(amount) AS total FROM orders GROUP BY customer),
r AS (
  SELECT customer, total, ROW_NUMBER() OVER (ORDER BY total DESC, customer) AS rn FROM t)
SELECT CASE WHEN rn <= 2 THEN customer ELSE 'Other' END AS label,
       SUM(total) AS revenue
FROM r
GROUP BY label
ORDER BY MIN(rn);

label | revenue
------+--------
Dee   | 500    
Asha  | 400    
Other | 560    
(3 rows)



## 17. Greatest-n-per-group without window functions
- Three classic ways, all work on old databases:
  1. correlated subquery with `MAX`
  2. join to a grouped `MAX` (a "derived table")
  3. `NOT EXISTS` (no row has a bigger one)
- Big gotcha: all three return **every** row tied for the max. Asha's two orders on 2024-02-10 both come back, unlike `ROW_NUMBER`.
- SQLite bonus: with a single `MAX()` in the select, bare columns come from the max row. Convenient, but SQLite-only, and ambiguous on ties (Asha gets 80 here, not 200: either is possible).

In [25]:
%%sql
-- 1. correlated subquery
SELECT customer, order_date, amount FROM orders o
WHERE order_date = (SELECT MAX(order_date) FROM orders WHERE customer = o.customer)
ORDER BY customer, id;

customer | order_date | amount
---------+------------+-------
Asha     | 2024-02-10 | 80    
Asha     | 2024-02-10 | 200   
Ben      | 2024-03-01 | 300   
Cy       | 2024-02-14 | 60    
Dee      | 2024-03-09 | 500   
(5 rows)



In [26]:
%%sql
-- 2. join to grouped max
SELECT o.customer, o.order_date, o.amount
FROM orders o
JOIN (SELECT customer, MAX(order_date) AS d FROM orders GROUP BY customer) m
  ON m.customer = o.customer AND m.d = o.order_date
ORDER BY o.customer, o.id;

customer | order_date | amount
---------+------------+-------
Asha     | 2024-02-10 | 80    
Asha     | 2024-02-10 | 200   
Ben      | 2024-03-01 | 300   
Cy       | 2024-02-14 | 60    
Dee      | 2024-03-09 | 500   
(5 rows)



In [27]:
%%sql
-- 3. NOT EXISTS: nobody is later than me
SELECT customer, order_date, amount FROM orders o
WHERE NOT EXISTS (SELECT 1 FROM orders x WHERE x.customer = o.customer AND x.order_date > o.order_date)
ORDER BY customer, id;

customer | order_date | amount
---------+------------+-------
Asha     | 2024-02-10 | 80    
Asha     | 2024-02-10 | 200   
Ben      | 2024-03-01 | 300   
Cy       | 2024-02-14 | 60    
Dee      | 2024-03-09 | 500   
(5 rows)



In [28]:
%%sql
-- SQLite bare-column trick: amount comes from the row holding the max date (one row per customer)
SELECT customer, MAX(order_date) AS last_date, amount FROM orders GROUP BY customer ORDER BY customer;

customer | last_date  | amount
---------+------------+-------
Asha     | 2024-02-10 | 80    
Ben      | 2024-03-01 | 300   
Cy       | 2024-02-14 | 60    
Dee      | 2024-03-09 | 500   
(4 rows)



## 18. Bad vs good: NOT IN with NULLs
- `x NOT IN (list with NULL)` is never true, so you silently get zero rows.
- Here the "list" contains a `NULL` email. `NOT EXISTS` behaves as you expect.

In [29]:
%%sql
CREATE TABLE vip (email TEXT);
INSERT INTO vip VALUES ('a@x.com'), (NULL);
-- Bad: returns zero rows because of the NULL
SELECT DISTINCT email FROM contact WHERE email NOT IN (SELECT email FROM vip) ORDER BY email;
-- Good: returns b@x.com and c@x.com
SELECT DISTINCT email FROM contact c
WHERE NOT EXISTS (SELECT 1 FROM vip v WHERE v.email = c.email) ORDER BY email;

email
-----
(0 rows)

email  
-------
b@x.com
c@x.com
(2 rows)



## 19. Keyset pagination (instead of OFFSET)
- `LIMIT 3 OFFSET 6` makes the database walk and throw away 6 rows. Page 1000 is slow. Also if a row is inserted meanwhile, rows shift and you see repeats or gaps.
- Keyset: remember the last row you showed, ask for "rows after it". With an index on the sort key it is fast on any page.
- The sort key must be unique, so combine columns: `(salary DESC, id)`.
- When every column sorts the same way, a row-value comparison does the "after" test: `(a, b) > (x, y)` (SQLite 3.15+). Our sort is mixed (`salary DESC`, `id` ascending), so below we spell the condition out as `a < x OR (a = x AND b > y)`.

In [30]:
%%sql
-- page 1
SELECT id, name, salary FROM emp ORDER BY salary DESC, id LIMIT 4;

id | name  | salary
---+-------+-------
1  | Alice | 9500  
8  | Finn  | 7500  
2  | Bob   | 7200  
3  | Gus   | 7200  
(4 rows)



In [31]:
%%sql
-- page 2: last row of page 1 was (salary 7200, id 3); ask for rows strictly after that
SELECT id, name, salary FROM emp
WHERE salary < 7200 OR (salary = 7200 AND id > 3)
ORDER BY salary DESC, id LIMIT 4;

id | name  | salary
---+-------+-------
5  | Carla | 6800  
11 | Kim   | 6800  
6  | Dev   | 6100  
4  | Hana  | 6000  
(4 rows)



In [32]:
%%sql
-- page 3: last row of page 2 was (6800, id 11)
SELECT id, name, salary FROM emp
WHERE salary < 6800 OR (salary = 6800 AND id > 11)
ORDER BY salary DESC, id LIMIT 4;

id | name | salary
---+------+-------
6  | Dev  | 6100  
4  | Hana | 6000  
7  | Eve  | 5800  
9  | Ivy  | 5000  
(4 rows)



## 20. Leaderboard with dense ranks and "points behind"
- A game-style board: tied players share a place, the next place does not skip (dense), and we show the gap to the leader.
- Show `RANK` next to `DENSE_RANK` so you can choose which style the product wants (sports often skip: 1, 2, 2, 4).

In [33]:
%%sql
CREATE TABLE score (player TEXT PRIMARY KEY, points INTEGER);
INSERT INTO score VALUES ('Zed',90),('Amy',85),('Raj',85),('Lee',70),('Mo',70),('Sam',60);
SELECT player, points,
       DENSE_RANK() OVER (ORDER BY points DESC) AS place_dense,
       RANK()       OVER (ORDER BY points DESC) AS place_sports,
       MAX(points) OVER () - points AS behind_leader
FROM score ORDER BY points DESC, player;

player | points | place_dense | place_sports | behind_leader
-------+--------+-------------+--------------+--------------
Zed    | 90     | 1           | 1            | 0            
Amy    | 85     | 2           | 2            | 5            
Raj    | 85     | 2           | 2            | 5            
Lee    | 70     | 3           | 4            | 20           
Mo     | 70     | 3           | 4            | 20           
Sam    | 60     | 4           | 6            | 30           
(6 rows)



## 21. Putting it together: best order per customer, with their share
- Combine patterns: rank orders per customer, keep the biggest, and show what share of the customer's total it is.
- Window aggregates and ranks can sit in the same CTE. Filter in the outer query.

In [34]:
%%sql
WITH r AS (
  SELECT customer, id, amount,
         ROW_NUMBER() OVER (PARTITION BY customer ORDER BY amount DESC, id) AS rn,
         SUM(amount)  OVER (PARTITION BY customer) AS cust_total
  FROM orders)
SELECT customer, id AS biggest_order, amount, cust_total,
       ROUND(100.0 * amount / cust_total, 1) AS pct_of_total
FROM r WHERE rn = 1 ORDER BY customer;

customer | biggest_order | amount | cust_total | pct_of_total
---------+---------------+--------+------------+-------------
Asha     | 3             | 200    | 400        | 50.0        
Ben      | 5             | 300    | 350        | 85.7        
Cy       | 6             | 75     | 210        | 35.7        
Dee      | 9             | 500    | 500        | 100.0       
(4 rows)

